# Feature Engineering — Exploration

This notebook explores how to build the feature set for the Version 1 machine learning model.

## Goal

Understand how each processed dataset connects, identify team name mismatches, and design the feature engineering pipeline before writing production code.

## Datasets Used

| File | Purpose |
|------|---------|
| `clean_results.csv` | Match history — recent form features |
| `fifa_rankings.csv` | FIFA rank and points |
| `elo_ratings.csv` | Elo team strength |
| `world_cup_history.csv` | Appearances, titles, best finish |
| `world_cup_fixtures.csv` | 2026 tournament match schedule |
| `former_names.csv` | Historical team name mappings |

In [ ]:
import pandas as pd
import numpy as np

results     = pd.read_csv("../data/processed/clean_results.csv", parse_dates=["date"])
fifa        = pd.read_csv("../data/processed/fifa_rankings.csv")
elo         = pd.read_csv("../data/processed/elo_ratings.csv")
wc_history  = pd.read_csv("../data/processed/world_cup_history.csv")
fixtures    = pd.read_csv("../data/processed/world_cup_fixtures.csv")
former      = pd.read_csv("../data/processed/former_names.csv")

print("results:    ", results.shape)
print("fifa:       ", fifa.shape)
print("elo:        ", elo.shape)
print("wc_history: ", wc_history.shape)
print("fixtures:   ", fixtures.shape)
print("former:     ", former.shape)

---

## 1. Team Name Coverage

Each dataset uses a different naming convention. Before building features, we need to understand how many teams overlap.

In [ ]:
results_teams = set(results['home_team']) | set(results['away_team'])
fifa_teams    = set(fifa['team'].dropna())
elo_codes     = set(elo['country_code'].dropna())
wc_teams      = set(wc_history['team'].dropna())
fixture_teams = set(fixtures['home_team'].dropna()) | set(fixtures['away_team'].dropna())

print(f"Teams in results:         {len(results_teams)}")
print(f"Teams in FIFA rankings:   {len(fifa_teams)}")
print(f"Codes  in Elo ratings:    {len(elo_codes)}")
print(f"Teams in WC history:      {len(wc_teams)}")
print(f"Teams in 2026 fixtures:   {len(fixture_teams)}")

In [ ]:
# All 48 fixture teams appear in FIFA rankings — no missing
missing_from_fifa = fixture_teams - fifa_teams
print(f"Fixture teams missing from FIFA rankings: {len(missing_from_fifa)}")

---

## 2. Former Names — Team Name Standardization

`former_names.csv` maps old team names (e.g. `West Germany`) to current names (e.g. `Germany`).

This is needed to merge historical results with modern datasets.

In [ ]:
former.head(10)

In [ ]:
name_map = dict(zip(former['former'], former['current']))

results['home_team_std'] = results['home_team'].replace(name_map)
results['away_team_std'] = results['away_team'].replace(name_map)

changed_home = (results['home_team'] != results['home_team_std']).sum()
changed_away = (results['away_team'] != results['away_team_std']).sum()
print(f"Home team names standardized: {changed_home}")
print(f"Away team names standardized: {changed_away}")

In [ ]:
# What names changed?
changed = results[results['home_team'] != results['home_team_std']][['home_team', 'home_team_std']].drop_duplicates()
changed.columns = ['former', 'current']
changed

---

## 3. Recent Form Features

For each team, calculate rolling statistics over their last 10 matches before each game:

- Win percentage
- Goals scored per match
- Goals conceded per match
- Goal difference
- Clean sheet percentage

In [ ]:
# Reshape results into one row per team per match
home = results[['date', 'home_team_std', 'away_team_std', 'home_score', 'away_score', 'neutral']].copy()
home.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'neutral']
home['is_home'] = True

away = results[['date', 'away_team_std', 'home_team_std', 'away_score', 'home_score', 'neutral']].copy()
away.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'neutral']
away['is_home'] = False

team_matches = pd.concat([home, away], ignore_index=True)
team_matches = team_matches.sort_values(['team', 'date']).reset_index(drop=True)

team_matches['win']         = (team_matches['goals_for'] > team_matches['goals_against']).astype(int)
team_matches['clean_sheet'] = (team_matches['goals_against'] == 0).astype(int)

print(f"Team-match rows: {len(team_matches):,}")
team_matches.head()

In [ ]:
# Compute rolling 10-match form (shift by 1 to avoid data leakage)
WINDOW = 10

def rolling_form(group):
    group = group.sort_values('date')
    shifted = group[['win', 'goals_for', 'goals_against', 'clean_sheet']].shift(1)
    rolled  = shifted.rolling(WINDOW, min_periods=1).mean()
    group['form_win_pct']          = rolled['win']
    group['form_goals_scored']     = rolled['goals_for']
    group['form_goals_conceded']   = rolled['goals_against']
    group['form_goal_diff']        = rolled['goals_for'] - rolled['goals_against']
    group['form_clean_sheet_pct']  = rolled['clean_sheet']
    return group

team_matches = team_matches.groupby('team', group_keys=False).apply(rolling_form)

print("Form features added.")
team_matches[team_matches.team == 'Argentina'][[
    'date', 'goals_for', 'goals_against',
    'form_win_pct', 'form_goals_scored', 'form_goals_conceded'
]].tail(10)

---

## 4. Join FIFA Rankings

FIFA rankings are a single snapshot (current). Join on team name. All 48 fixture teams are covered.

In [ ]:
fifa_lookup = fifa[['team', 'fifa_rank', 'fifa_points', 'confederation']].copy()

sample_join = pd.DataFrame({'team': sorted(fixture_teams)}).merge(fifa_lookup, on='team', how='left')
print(f"Matched: {sample_join.fifa_rank.notna().sum()} / {len(sample_join)}")
sample_join.head(8)

---

## 5. Join Elo Ratings — Code Mismatch Found

**Finding:** FIFA uses 3-letter ISO codes (e.g. `ARG`, `ESP`). Elo uses its own 2-letter codes (e.g. `AR`, `ES`).

Simple truncation (first 2 chars) only matches 181 of 211 teams because many codes diverge (e.g. FIFA `POR` → Elo `PT`, FIFA `SUI` → Elo `CH`).

**Decision:** Build a curated `data/raw/elo_code_map.csv` that maps FIFA 3-letter codes to Elo 2-letter codes. This is a one-time manual file that covers all 48 World Cup teams.

In [ ]:
# Demonstrate the mismatch
print("FIFA country_id sample:")
print(fifa[['team','country_id']].head(10).to_string(index=False))
print("\nElo country_code sample:")
print(elo[['country_code','elo_rating']].head(10).to_string(index=False))

In [ ]:
# Identify fixture teams that need manual code mapping
# (any team where FIFA 3-letter != Elo 2-letter pattern)
test = fifa[fifa['team'].isin(fixture_teams)][['team','country_id']].copy()
test['elo_code_try'] = test['country_id'].str[:2]
merged_test = test.merge(elo[['country_code','elo_rating']], left_on='elo_code_try', right_on='country_code', how='left')
unmatched = merged_test[merged_test.elo_rating.isna()][['team','country_id']]
print(f"Fixture teams needing manual Elo code mapping: {len(unmatched)}")
print(unmatched.to_string(index=False))

---

## 6. World Cup History — Name Mismatches Found

**Finding:** 11 fixture teams are not in `world_cup_history.csv` under their FIFA name.

Some are genuine first-timers (Cabo Verde, Curaçao, Uzbekistan, Jordan). Others exist under different names:

| Fixture Name | WC History Name |
|---|---|
| Czechia | Czech Republic |
| Korea Republic | South Korea |
| USA | United States |
| IR Iran | Iran |
| Türkiye | Turkey |
| Côte d'Ivoire | Ivory Coast |
| Congo DR | DR Congo |

**Decision:** Build a `fixture_to_wc_name_map` dictionary in `build_features.py`. Teams with no WC history get zeros (appearances=0, titles=0, best_finish=8 for group stage).

In [ ]:
# Teams missing from WC history
not_in_wc = sorted(fixture_teams - set(wc_history['team']))
print("Fixture teams not in WC history:")
for t in not_in_wc:
    print(f"  {t}")

In [ ]:
# Apply name fixes and check coverage
fixture_to_wc = {
    'Czechia':         'Czech Republic',
    'Korea Republic':  'South Korea',
    'USA':             'United States',
    'IR Iran':         'Iran',
    'Türkiye':         'Turkey',
    "Côte d'Ivoire":   'Ivory Coast',
    'Congo DR':        'DR Congo',
}

# Verify fixes exist in WC history
for fixture_name, wc_name in fixture_to_wc.items():
    found = wc_name in set(wc_history['team'])
    print(f"  '{fixture_name}' -> '{wc_name}': {'✅ found' if found else '❌ not found'}")

---

## 7. Summary — What the Feature Pipeline Needs to Do

Based on this exploration, the `build_features.py` script will:

1. Apply `former_names.csv` to standardize team names in `results`.
2. Reshape results into one row per team per match.
3. Compute rolling 10-match form features per team per match (with shift to avoid leakage).
4. Join FIFA rankings by team name.
5. Join Elo ratings using a curated `elo_code_map.csv` (FIFA 3-letter → Elo 2-letter).
6. Join World Cup history using `fixture_to_wc` name map. Teams with no WC history get zeros.
7. Reconstruct match-level rows: one row = one match, with home and away features side by side.
8. Add match context features: host nation, neutral venue, confederation.
9. Save final training dataset to `data/processed/features.csv`.

### Outstanding items before `build_features.py`

- **Create `data/raw/elo_code_map.csv`** — curated FIFA 3-letter to Elo 2-letter code mapping for all 48 fixture teams.
- Remaining 4 truly first-time WC qualifiers (Cabo Verde, Curaçao, Uzbekistan, Jordan) get all-zero WC history.